In [235]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from statsmodels.formula.api import mixedlm
from statsmodels.stats.multitest import multipletests
from sklearn.preprocessing import StandardScaler
cwd = os.getcwd()
data_dir = os.path.join(cwd, 'data')
out_dir = os.path.join(cwd, 'output')

In [236]:
def get_earliest_row(df, date_col):
    # Ensure the column is datetime
    df[date_col] = pd.to_datetime(df[date_col])
    
    # Get index of earliest date
    idx = df[date_col].idxmin()
    
    # Return that row
    return df.loc[idx]
def delta_months(date1, date2):
    d1 = pd.to_datetime(date1)
    d2 = pd.to_datetime(date2)
    
    return (d2.year - d1.year) * 12 + (d2.month - d1.month)

In [259]:
# Load data
raw_data_df = pd.read_csv(os.path.join(data_dir, 'MDS-UPDRS_Part_III_24Jan2024.csv'))

# MEDSTATE
raw_data_df['MEDSTATE'] = None
raw_data_df.loc[raw_data_df['PDSTATE'] == 'ON', 'MEDSTATE'] = 'ON'
raw_data_df.loc[
    (raw_data_df['PDSTATE'] == 'OFF') |
    (raw_data_df['PDSTATE'].isna() & (raw_data_df['PDTRTMNT'] == 0)),
    'MEDSTATE'
] = 'OFF'

# Keep relevant columns
columns_of_interest = ['PATNO', 'EVENT_ID', 'INFODT', 'MEDSTATE', 'NP3TOT', 'NHY'] + raw_data_df.columns.tolist()[23:-7]
df = raw_data_df[columns_of_interest].dropna().copy()

# Subscores
df['RIGIDITY'] = df[['NP3RIGN', 'NP3RIGRU', 'NP3RIGLU', 'NP3RIGRL', 'NP3RIGLL']].sum(axis=1)

df['BRADY'] = df[
    ['NP3FTAPR', 'NP3FTAPL', 'NP3HMOVR', 'NP3HMOVL', 'NP3PRSPR',
     'NP3PRSPL', 'NP3TTAPR', 'NP3TTAPL', 'NP3LGAGR', 'NP3LGAGL', 'NP3BRADY']
].sum(axis=1)

df['PIGD'] = df[['NP3RISNG', 'NP3GAIT', 'NP3FRZGT', 'NP3PSTBL', 'NP3POSTR']].sum(axis=1)

df['TREMOR'] = df[
    ['NP3PTRMR', 'NP3PTRML', 'NP3KTRMR', 'NP3KTRML', 'NP3RTARU',
     'NP3RTALU', 'NP3RTARL', 'NP3RTALL', 'NP3RTALJ', 'NP3RTCON']
].sum(axis=1)

# Visit month
visit_dict = {
    'BL': 0, 'V01': 3, 'V02': 6, 'V03': 9, 'V04': 12,
    'V05': 18, 'V06': 24, 'V07': 30, 'V08': 36, 'V09': 42,
    'V10': 48, 'V11': 54, 'V12': 60, 'V13': 72, 'V14': 84,
    'V15': 96, 'V16': 108, 'V17': 120, 'V18': 132, 'V19': 144,
    'V20': 156
}
df['MONTH'] = df['EVENT_ID'].map(visit_dict)

# Keep only first 12 months
df = df[df['MONTH'] <= 12].copy()

# Keep only patients with at least 5 visits
df = df[df.groupby('PATNO')['PATNO'].transform('size') >= 5].copy()

# Baseline NHY from rows where MONTH == 0
baseline_nhy = (
    df[df['MONTH'] == 0]
    .drop_duplicates('PATNO')
    .set_index('PATNO')['NHY']
)
df['NHY_BASELINE'] = df['PATNO'].map(baseline_nhy)

# Define stage from baseline NHY
df['STAGE'] = np.where(df['NHY_BASELINE'] >= 2, 'LATE', 'EARLY')

# saving master df
df.to_csv(os.path.join(data_dir, 'master_df.csv'), index=False)

print(f"Total entries: {len(df)}")
print(f"Total unique patients: {df['PATNO'].nunique()}")

df


Total entries: 2616
Total unique patients: 513


,PATNO,EVENT_ID,INFODT,MEDSTATE,NP3TOT,NHY,NP3SPCH,NP3FACXP,NP3RIGN,NP3RIGRU,...,NP3RTALL,NP3RTALJ,NP3RTCON,RIGIDITY,BRADY,PIGD,TREMOR,MONTH,NHY_BASELINE,STAGE
9,3001,BL,03/2011,OFF,12.0,1.0,1.0,1.0,0.0,2.0,...,0.0,0.0,0.0,2.0,5.0,1.0,2.0,0.0,1.0,EARLY
11,3001,V01,05/2011,OFF,18.0,2.0,1.0,1.0,0.0,2.0,...,0.0,0.0,0.0,4.0,7.0,1.0,4.0,3.0,1.0,EARLY
12,3001,V02,08/2011,OFF,23.0,2.0,1.0,1.0,0.0,2.0,...,0.0,0.0,0.0,4.0,9.0,1.0,7.0,6.0,1.0,EARLY
13,3001,V03,11/2011,OFF,19.0,2.0,0.0,2.0,0.0,2.0,...,0.0,0.0,0.0,3.0,7.0,1.0,6.0,9.0,1.0,EARLY
14,3001,V04,03/2012,OFF,20.0,2.0,1.0,1.0,0.0,2.0,...,0.0,0.0,0.0,3.0,8.0,1.0,6.0,12.0,1.0,EARLY
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24125,182341,BL,01/2023,OFF,8.0,1.0,1.0,1.0,0.0,0.0,...,0.0,0.0,0.0,1.0,4.0,0.0,1.0,0.0,1.0,EARLY
24126,182341,V02,07/2023,ON,14.0,1.0,0.0,1.0,0.0,0.0,...,0.0,0.0,1.0,1.0,5.0,3.0,4.0,6.0,1.0,EARLY
24127,182341,V02,07/2023,OFF,13.0,1.0,1.0,1.0,0.0,0.0,...,0.0,0.0,0.0,1.0,6.0,2.0,2.0,6.0,1.0,EARLY
24128,182341,V04,01/2024,ON,27.0,2.0,1.0,2.0,0.0,0.0,...,0.0,0.0,4.0,1.0,13.0,2.0,8.0,12.0,1.0,EARLY


In [260]:
# Plotting for visualizing results
%matplotlib qt

# Checking sample size for each stage
early_df = df[df['STAGE'] == 'EARLY']
late_df = df[df['STAGE'] == 'LATE']

print(f"Early stage entries: {len(early_df)} \nEarly stage unique patients: {early_df['PATNO'].nunique()}")
print(f"Late stage entries: {len(late_df)} \nLate stage unique patients: {late_df['PATNO'].nunique()}")


Early stage entries: 1262 
Early stage unique patients: 248
Late stage entries: 1354 
Late stage unique patients: 265


In [261]:
# PLotting for visualizing results (regreesion with 95%CI)
for measure in ['NP3TOT', 'RIGIDITY', 'BRADY', 'PIGD', 'TREMOR']:
    plt.figure(figsize=(10, 6))

    # Early
    sns.regplot(
        data=early_df, x='MONTH', y=measure,
        scatter=False,  # removes scatter
        ci=95,
        label='Early Stage',
        color='b',
        line_kws={'linewidth': 2}
    )


    # Late
    sns.regplot(
        data=late_df, x='MONTH', y=measure,
        scatter=False,
        ci=95,
        label='Late Stage',
        color='r',
        line_kws={'linewidth': 2}
    )

    plt.xlabel('Months Since Baseline')
    plt.ylabel(f'{measure} Score')
    plt.title(f'{measure} Progression Over Time')
    plt.legend()
    plt.show()

In [262]:
def multiple_dvs_lmm(df, dvs, ivs = ' ~ MONTHS_SINCE_BASELINE*STAGE', 
                     random_intercept='PATNO', random_slope='~MONTH', std_dv = False):
    """
    Fit linear mixed models (LMMs) for multiple dependent variables (DVs).

    Parameters:
    df (pd.DataFrame): DataFrame containing the data for modeling.
    dvs (list of str): List of dependent variable column names to model.
    ivs (str): Fixed effects formula (default includes Occasion, Age, and Sex).
    random_intercept (str): Column name to use for random intercept grouping (e.g., participant ID).
    random_slope (str): Random slope formula (default is '~Occasion').

    Returns:
    list: List of fitted LMM model results (MixedLMResults objects).
    """
    mdfs = []
    scaler = StandardScaler()

    for dv in dvs:

        # Standardize the dependent variable if specified
        if std_dv:
            standardized_col = dv + "_z"
            df[standardized_col] = scaler.fit_transform(df[[dv]])
            formula = standardized_col + ivs
        else:
            formula = dv + ivs # Construct the full formula: e.g., 'Measure ~ Occasion + Age + Sex'
        
        md = mixedlm(formula, data=df, groups=df[random_intercept], re_formula=random_slope)
        mdf = md.fit(method='lbfgs')
        mdfs.append(mdf)

    return mdfs

def univariate_lmm_dataframe(dvs, mdfs, output=None, save=False):
    """
    Create a summary DataFrame of parameter estimates and p-values for each DV's LMM.

    Parameters:
    dvs (list of str): List of dependent variable names corresponding to the models.
    mdfs (list): List of fitted LMM results.
    output (str): File path to save the output Excel file (if save=True).
    save (bool): Whether to save the summary DataFrame to an Excel file.

    Returns:
    pd.DataFrame: Multi-index DataFrame summarizing parameters and p-values per DV.
    """
    columns  = []
    ivs = mdfs[0].pvalues.dropna().index # Assume all models have the same IVs
    for iv in ivs:
        columns.append((iv, 'param')) # Parameter estimate
        columns.append((iv,'pval')) # p-value
        columns.append((iv, 'pval_fdr')) # FDR-corrected p-value
    
    df = pd.DataFrame(index=dvs, columns=pd.MultiIndex.from_tuples(columns))
    for dv, mdf in zip(dvs, mdfs):
        for iv in ivs:
            df.loc[dv, (iv, 'param')] = round(mdf.params[iv],3)
            df.loc[dv, (iv, 'pval')] = round(mdf.pvalues[iv],3)

        # Apply BH-FDR correction per IV across DVs
    for iv in ivs:
        # Convert to numeric, force None -> NaN
        pvals = pd.to_numeric(df[(iv, 'pval')], errors='coerce')

        # Mask valid p-values
        valid_mask = pvals.notna()

        # Initialize corrected p-values with NaN
        pvals_fdr = np.full(len(pvals), np.nan)

        # Apply FDR only to valid p-values
        if valid_mask.sum() > 0:
            _, pvals_fdr_valid, _, _ = multipletests(
                pvals[valid_mask].values,
                method='fdr_bh'
            )
            pvals_fdr[valid_mask] = pvals_fdr_valid

        # Assign back to DataFrame
        df[(iv, 'pval_fdr')] = pvals_fdr

        # Round raw p-values last
        df[(iv, 'pval')] = pvals
    
    if save:
        df.to_excel(output)
    return df

In [263]:
mdfs = multiple_dvs_lmm(df, dvs=['NP3TOT', 'RIGIDITY', 'BRADY', 'PIGD', 'TREMOR'], ivs=' ~ MONTH*STAGE')
summary_df = univariate_lmm_dataframe(dvs=['NP3TOT', 'RIGIDITY', 'BRADY', 'PIGD', 'TREMOR'], mdfs=mdfs, output=os.path.join(out_dir, 'lmm_summary.xlsx'), save=True)
summary_df

c:\Users\g.acevedo\AppData\Local\anaconda3\Lib\site-packages\statsmodels\regression\mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)


Intercept               STAGE[T.LATE]                MONTH         \
             param pval pval_fdr         param pval pval_fdr  param   pval   
NP3TOT      11.855  0.0      0.0        12.409  0.0      0.0  0.205  0.000   
RIGIDITY     1.922  0.0      0.0         2.408  0.0      0.0  0.044  0.000   
BRADY        5.011  0.0      0.0         6.715  0.0      0.0  0.087  0.001   
PIGD         1.035  0.0      0.0         0.904  0.0      0.0  0.027  0.001   
TREMOR       2.942  0.0      0.0         1.386  0.0      0.0  0.027  0.045   

                  MONTH:STAGE[T.LATE]  ...           Group Var                \
         pval_fdr               param  ...  pval_fdr     param pval pval_fdr   
NP3TOT    0.00000              -0.142  ...  0.041667     1.862  0.0      0.0   
RIGIDITY  0.00000              -0.023  ...  0.192000     2.221  0.0      0.0   
BRADY     0.00125              -0.081  ...  0.041667     1.977  0.0      0.0   
PIGD      0.00125               0.014  ...  0.192000     1.725  0.0      0.0   
TREMOR    0.04500              -0.045  ...  0.041667      2.42  0.0      0.0   

         Group x MONTH Cov                 MONTH Var                
                     param   pval pval_fdr     param pval pval_fdr  
NP3TOT               0.005  0.567  0.70875     0.008  0.0      0.0  
RIGIDITY            -0.027  0.013  0.03250     0.008  0.0      0.0  
BRADY               -0.024  0.024  0.04000     0.009  0.0      0.0  
PIGD                 0.032  0.000  0.00000     0.009  0.0      0.0  
TREMOR                -0.0  0.969  0.96900     0.004  0.0      0.0  

[5 rows x 21 columns]

In [264]:
# Full cohort
mdfs = multiple_dvs_lmm(df, dvs=['NP3TOT', 'RIGIDITY', 'BRADY', 'PIGD', 'TREMOR'], 
                        ivs=' ~ MONTH', random_intercept='PATNO', random_slope='~MONTH', std_dv=False)
results_all = univariate_lmm_dataframe(dvs=['NP3TOT', 'RIGIDITY', 'BRADY', 'PIGD', 'TREMOR'], 
                                      mdfs=mdfs, output=os.path.join(out_dir, 'lmm_all.xlsx'), 
                                      save=True)

# Early stage
mdfs_early = multiple_dvs_lmm(early_df, dvs=['NP3TOT', 'RIGIDITY', 'BRADY', 'PIGD', 'TREMOR'], 
                        ivs=' ~ MONTH', random_intercept='PATNO', random_slope='~MONTH', std_dv=False)
results_early = univariate_lmm_dataframe(dvs=['NP3TOT', 'RIGIDITY', 'BRADY', 'PIGD', 'TREMOR'], 
                                      mdfs=mdfs_early, output=os.path.join(out_dir, 'lmm_early.xlsx'), 
                                      save=True)

# Late stage
mdfs_late = multiple_dvs_lmm(late_df, dvs=['NP3TOT', 'RIGIDITY', 'BRADY', 'PIGD', 'TREMOR'], 
                        ivs=' ~ MONTH', random_intercept='PATNO', random_slope='~MONTH', std_dv=False)
results_late = univariate_lmm_dataframe(dvs=['NP3TOT', 'RIGIDITY', 'BRADY', 'PIGD', 'TREMOR'], 
                                      mdfs=mdfs_late, output=os.path.join(out_dir, 'lmm_late.xlsx'), 
                                      save=True)

c:\Users\g.acevedo\AppData\Local\anaconda3\Lib\site-packages\statsmodels\regression\mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)
c:\Users\g.acevedo\AppData\Local\anaconda3\Lib\site-packages\statsmodels\regression\mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)
c:\Users\g.acevedo\AppData\Local\anaconda3\Lib\site-packages\statsmodels\regression\mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)
